In [202]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Conv2D #Conv2D extracts the features
from tensorflow.keras.layers import Flatten #Flatten reshapes them

from tensorflow.keras.layers import MaxPooling2D #MaxPooling2D reduces size
from tensorflow.keras.layers import Dropout #Dropout prevents overfitting

from tensorflow.keras.utils import to_categorical #converts numeric class labels to one hot encoded format for training classification models

In [203]:
df = pd.read_csv("mnist_train.csv")
df_test = pd.read_csv("mnist_test.csv")

In [204]:
df.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [205]:
df_test.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,NaN
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,NaN
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,NaN
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,NaN
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,NaN


In [206]:
df.shape


(42000, 785)

In [207]:
df_test.shape

(28000, 785)

In [208]:
df.columns

Index(['label', 'pixel0', 'pixel1', 'pixel2', 'pixel3', 'pixel4', 'pixel5',
       'pixel6', 'pixel7', 'pixel8',
       ...
       'pixel774', 'pixel775', 'pixel776', 'pixel777', 'pixel778', 'pixel779',
       'pixel780', 'pixel781', 'pixel782', 'pixel783'],
      dtype='str', length=785)

In [209]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 42000 entries, 0 to 41999
Columns: 785 entries, label to pixel783
dtypes: int64(785)
memory usage: 251.5 MB


In [210]:
df.isnull().sum()

label       0
pixel0      0
pixel1      0
pixel2      0
pixel3      0
           ..
pixel779    0
pixel780    0
pixel781    0
pixel782    0
pixel783    0
Length: 785, dtype: int64

In [211]:
#preprocessing
X_train = df.drop("label", axis=1).values
y_train = df["label"].values
X_test = df_test.drop("label", axis=1).values
y_test = df_test["label"].values

In [212]:
#normalize the value by diving each pixel by 255
X_train = X_train.astype("float32")/255.0
X_test = X_test.astype("float32")/255.0

In [213]:
#reshaping the data
X_train_reshape = X_train.reshape(-1, 28, 28) #-1 will be th first row and 28x28 will be the size of the image
X_test_reshape = X_test.reshape(-1, 28, 28)

In [214]:
# now we convert those labels(0-9) into categorical form
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

In [215]:
#perceptron
peceptron = Sequential([   
    Flatten(input_shape=(28,28)),
    Dense(10, activation="softmax")
])

In [216]:
peceptron.compile(optimizer="sgd", loss="categorical_crossentropy", metrics=["accuracy"])

In [217]:
history_perceptron = peceptron.fit(X_train_reshape, y_train_cat, epochs=5,
    batch_size=32, validation_data=(X_test_reshape, y_test_cat), verbose=1)

Epoch 1/5
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7984 - loss: 0.8674 - val_accuracy: 1.0000 - val_loss: nan
Epoch 2/5
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.8744 - loss: 0.4954 - val_accuracy: 1.0000 - val_loss: nan
Epoch 3/5
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8866 - loss: 0.4302 - val_accuracy: 1.0000 - val_loss: nan
Epoch 4/5
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.8930 - loss: 0.3980 - val_accuracy: 1.0000 - val_loss: nan
Epoch 5/5
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8970 - loss: 0.3778 - val_accuracy: 1.0000 - val_loss: nan


In [218]:
acc_perc = peceptron.evaluate(X_test_reshape, y_test_cat, verbose=0)[1]

In [219]:
acc_perc

1.0

In [220]:
#now using ann
ann = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(10, activation="softmax")
])

In [221]:
ann.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [222]:
history_ann = ann.fit(X_train_reshape, y_train_cat, epochs=5,
                      batch_size=32, validation_data=(X_test_reshape, y_test_cat), verbose=1)

Epoch 1/5
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9202 - loss: 0.2759 - val_accuracy: 0.0000e+00 - val_loss: 2.4820
Epoch 2/5
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.9645 - loss: 0.1169 - val_accuracy: 0.0000e+00 - val_loss: 2.5254
Epoch 3/5
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9756 - loss: 0.0794 - val_accuracy: 0.0000e+00 - val_loss: 2.5802
Epoch 4/5
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9812 - loss: 0.0590 - val_accuracy: 0.0000e+00 - val_loss: 2.5998
Epoch 5/5
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9854 - loss: 0.0456 - val_accuracy: 0.0000e+00 - val_loss: 2.6271


In [223]:
acc_ann = ann.evaluate(X_test_reshape, y_test_cat, verbose=0)[1]

In [224]:
acc_ann

0.0

In [225]:
X_train_cnn = X_train.reshape(-1, 28, 28, 1)
X_test_cnn = X_test.reshape(-1, 28, 28, 1)

In [226]:
#cnn
cnn = Sequential([
    Conv2D(32, kernel_size=(3,3), activation="relu", input_shape=(28, 28, 1)),
    MaxPooling2D(pool_size=(2,2)),
    Conv2D(64, kernel_size=(3,3), activation="relu"),
    MaxPooling2D(pool_size=(2,2)),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(10, activation="softmax")
])

In [227]:
cnn.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [ ]:
cnn_history = cnn.fit(X_train_cnn, y_train_cat, epochs=5,
                      batch_size=32, validation_data = (X_test_cnn, y_test_cat), verbose=1)

Epoch 1/5
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 21s 15ms/step - accuracy: 0.9184 - loss: 0.2704 - val_accuracy: 0.0962 - val_loss: 12.8243
Epoch 2/5
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 20s 15ms/step - accuracy: 0.9712 - loss: 0.0976 - val_accuracy: 0.0967 - val_loss: 14.1003
Epoch 3/5
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 20s 15ms/step - accuracy: 0.9783 - loss: 0.0719 - val_accuracy: 0.0970 - val_loss: 16.2940
Epoch 4/5
1312/1313 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9835 - loss: 0.0561

In [ ]:
cnn_acc = cnn.evaluate(X_test_cnn, y_test_cat, verbose=0)[1]

In [ ]:
cnn_acc

0.09989285469055176

In [ ]:
def plot_training(history, title):
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(history.history['accuracy'], label="Train")
    plt.plot(history.history['val_accuracy'], label="Val")
    plt.title(f"{title} Accuracy")
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(history.history['loss'], label="Train")
    plt.plot(history.history['val_loss'], label="Val")
    plt.title(f"{title} Loss")
    plt.legend()
    plt.show()

In [ ]:
plot_training(history_perceptron, "Perceptron")

In [ ]:

plot_training(history_ann, "ANN")

In [ ]:
plot_training(cnn_history, "CNN")

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(history_perceptron.history['val_accuracy'], label="Perceptron")
plt.plot(history_ann.history['val_accuracy'], label="ANN")
plt.plot(cnn_history.history['val_accuracy'], label="CNN")
plt.title("Validation Accuracy Comparison")
plt.xlabel("Epochs")
plt.ylabel("Val Accuracy")
plt.legend()
plt.show()

In [ ]:
def show_side_by_side(models, model_names, X, X_cnn, y_true, n=5):
    idxs = np.random.choice(len(X), n, replace=False)
    plt.figure(figsize=(15, 6))
    for i, idx in enumerate(idxs):
        plt.subplot(2, n, i+1)
        plt.imshow(X[idx].reshape(28, 28), cmap="gray")
        plt.axis("off")
        plt.title(f"True: {y_true[idx]}")
        preds = [np.argmax(model.predict(X_cnn[idx].reshape(1, 28, 28, 1) if name == "CNN" else X[idx].reshape(1, 28, 28)))
                 for model, name in zip(models, model_names)]
        plt.subplot(2, n, n+i+1)
        plt.axis("off")
        plt.title("\n".join(f"{n}: {p}" for n, p in zip(model_names, preds)))
    plt.tight_layout()
    plt.show()

In [ ]:
show_side_by_side([peceptron, ann, cnn], ["Perceptron", "ANN", "CNN"], X_test_reshape, X_test_cnn, y_test, 5)

In [ ]:
y_pred_cnn = np.argmax(cnn.predict(X_test_cnn), axis=1)
cm = confusion_matrix(y_test, y_pred_cnn)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("CNN Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

In [ ]:

final_accs = [acc_perc*100, acc_ann*100, cnn_acc*100]
models = ["Perceptron", "ANN", "CNN"]

plt.figure(figsize=(8,6))
bars = plt.bar(models, final_accs, color=['#ff9999','#66b3ff','#99ff99'])
plt.title("Final Test Accuracy Comparison")
plt.ylabel("Accuracy (%)")
for bar, acc in zip(bars, final_accs):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()-1, f"{acc:.2f}%",
             ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.ylim(80, 100)
plt.show()